# The surprise against the reaction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Interest-Rate-Decision/notebooks/blob/main/02-surprise-vs-reaction.ipynb)

Does the pair move more when the decision is further from what was expected? The surprise is
**actual minus consensus**: a positive surprise is hawkish (a higher rate than expected).

**The consensus is not in InterestRateDecision.com's open data.** The site shows it on each decision's page, from
FXStreet's calendar, but that figure is not ours to redistribute, so `decisions.csv` has the decision and the rate only.
This notebook gives you two choices:

1. **Your own consensus** (best): a file `consensus.csv` with the columns `bank,date,consensus` (the expected rate in percent,
   from a poll or calendar you are allowed to use). The surprise is then `(rate − consensus) × 100` basis points.
2. **The move as a proxy** (what runs without it): the change against the previous rate, `change_bp`.
   **This is not a surprise.** Most moves are expected days ahead, so the proxy mixes the expected with the unexpected and
   will understate the link. Every chart says which one it uses.

The reaction is the move in pips 15 and 60 minutes after the announcement, turned so that **up means the bank's currency rose**.

### Candles: download your own

Dukascopy's candles are not ours to share, so this repository ships none (only a synthetic sample). Two ways to get them:

1. **The `dukascopy-node` CLI** (needs Node.js): `python scripts/download_candles.py --banks fed ecb --since 2020-01-01` runs
   `npx dukascopy-node -i eurusd -from ... -to ... -t m1 -f csv` for the hours around each decision and writes `candles/EURUSD/`.
   Add `--dry-run` to see the commands first.
2. **By hand** from Dukascopy's historical data export (one-minute, bid, GMT): save each CSV in `candles/<PAIR>/`.

`ird.read_candle_csv` reads both formats. Then set `SAMPLE = False` below.


In [ ]:
# On Colab or Kaggle, fetch the repository first: the helper module and the synthetic sample.
import os, subprocess, sys
if not os.path.exists("ird_events.py"):
    if not os.path.exists("notebooks-repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Interest-Rate-Decision/notebooks.git", "notebooks-repo"], check=True)
    os.chdir("notebooks-repo")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ird_events as ird

In [ ]:
SAMPLE = True                    # True: synthetic candles from data/sample, offline. False: your own downloads.
CANDLES_DIR = "candles"          # scripts/download_candles.py writes candles/<PAIR>/*.csv
BANKS = ["fed", "ecb", "boe"]    # any of: fed ecb boe boj boc rba rbnz snb norges riksbank
SINCE = "2015-01-01"             # IRD's decisions go back to 2008

ird.banner(SAMPLE)
decisions = ird.load_decisions(sample=SAMPLE)
decisions = decisions[decisions.bank.isin(BANKS) & (decisions.time_utc >= pd.Timestamp(SINCE, tz="UTC"))]
candles = {
    pair: ird.load_sample_candles(pair) if SAMPLE else ird.load_candles(pair, CANDLES_DIR)
    for pair in decisions.pair.dropna().unique()
}
print(len(decisions), "decisions;", {pair: len(c) for pair, c in candles.items()}, "one-minute candles")

In [ ]:
CONSENSUS_FILE = "consensus.csv"  # bank,date,consensus  e.g.  fed,2026-09-16,3.75

if os.path.exists(CONSENSUS_FILE):
    consensus = pd.read_csv(CONSENSUS_FILE, dtype={"date": str})
    data = decisions.merge(consensus, on=["bank", "date"], how="inner")
    data["surprise_bp"] = ((data["rate"] - data["consensus"]) * 100).round(1)
    SURPRISE_LABEL = "surprise: actual − consensus, bp"
else:
    data = decisions.copy()
    data["surprise_bp"] = data["change_bp"]
    SURPRISE_LABEL = "PROXY: the move against the previous rate, bp (not a surprise)"
print(SURPRISE_LABEL, "|", len(data), "decisions")

In [ ]:
rows = []
for d in data.itertuples():
    p = ird.path_in_pips(candles[d.pair], d.time_utc, d.pair, ird.BANKS[d.bank]["side"])
    if p is None:
        continue
    rows.append({"bank": d.bank, "date": d.date, "move": d.move, "surprise_bp": d.surprise_bp,
                 "pips15": p.loc[14], "pips60": p.loc[59]})
reactions = pd.DataFrame(rows)
reactions.head()

## Scatter, by bank

Each dot is a decision; the line is a least-squares fit, drawn only with two different surprises or more.

In [ ]:
banks = list(reactions.bank.unique())
fig, axes = plt.subplots(1, len(banks), figsize=(5 * len(banks), 4), squeeze=False)
for ax, bank in zip(axes[0], banks):
    part = reactions[reactions.bank == bank]
    ax.scatter(part.surprise_bp, part.pips15, alpha=0.7)
    if part.surprise_bp.nunique() >= 2:
        slope, intercept = np.polyfit(part.surprise_bp, part.pips15, 1)
        xs = np.linspace(part.surprise_bp.min(), part.surprise_bp.max(), 2)
        ax.plot(xs, slope * xs + intercept, color="black", lw=1, label=f"{slope:.2f} pips per bp")
        ax.legend()
    ax.axhline(0, color="black", lw=0.5, alpha=0.4)
    ax.set_title(f"{ird.BANKS[bank]['name']} on {ird.BANKS[bank]['pair']}")
    ax.set_xlabel(SURPRISE_LABEL)
    ax.set_ylabel(f"pips after 15 minutes (up = {ird.BANKS[bank]['currency']} stronger)")
fig.tight_layout()
plt.show()

## The numbers

Correlation and slope per bank, and the average reaction for each size of surprise.

In [ ]:
def fit(part):
    if len(part) < 3 or part.surprise_bp.nunique() < 2:
        return pd.Series({"n": len(part), "corr15": np.nan, "slope15": np.nan, "corr60": np.nan})
    return pd.Series({
        "n": len(part),
        "corr15": part.surprise_bp.corr(part.pips15),
        "slope15": np.polyfit(part.surprise_bp, part.pips15, 1)[0],
        "corr60": part.surprise_bp.corr(part.pips60),
    })

print(SURPRISE_LABEL)
display(reactions.groupby("bank")[["surprise_bp", "pips15", "pips60"]].apply(fit).round(2).astype({"n": int}))
reactions.groupby(["bank", "surprise_bp"])[["pips15", "pips60"]].agg(["mean", "count"]).round(1)

## Reading it

- With the proxy, a flat line does not mean surprises do not matter: it means the move alone does not say much, because most moves were expected. Bring a consensus file for the real test.
- The sign should be positive: a hawkish surprise lifts the bank's currency. When it is not, the statement, the press conference or the projections usually said more than the rate.
- InterestRateDecision.com counts how often each bank surprised, decision by decision since 2008, in its [study of surprises](https://interestratedecision.com/blog/how-often-do-central-banks-surprise/?via=notebooks).

---

Decisions from [InterestRateDecision.com](https://interestratedecision.com/?via=notebooks) (free to reuse with a link). The method: [how the site measures a reaction](https://interestratedecision.com/about/?via=notebooks#method). The studies: [how often central banks surprise](https://interestratedecision.com/blog/how-often-do-central-banks-surprise/?via=notebooks), [the most volatile forex news](https://interestratedecision.com/blog/most-volatile-forex-news/?via=notebooks), [how to trade FOMC](https://interestratedecision.com/blog/how-to-trade-fomc/?via=notebooks), [can central bank decisions be predicted?](https://interestratedecision.com/blog/can-central-bank-decisions-be-predicted/?via=notebooks). Learn to test and trade the news: [Forexia's courses](https://forexiapro.com/courses?via=notebooks).

Not financial advice. Past reactions say nothing certain about the next one.

Part of the Forexia network: [InterestRateDecision.com](https://interestratedecision.com/?via=notebooks) · [Forexia Academy](https://forexiapro.com/courses?via=notebooks) · [TradingSignalBot](https://tradingsignalbot.com/?via=notebooks) · [PropSim](https://github.com/PropSim-Enterprise) · [OpenCharts](https://github.com/dylanpersonguy/OpenCharts)